In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score


# 1. Load datasets

train_df = pd.read_csv("train.csv", engine='python', on_bad_lines='skip')
test_df = pd.read_csv("test.csv", engine='python', on_bad_lines='skip')



# 2. merge in game overview information
#    This can help performance by giving more text

# train_df = train_df.merge(overview_df, on="title", how="left")
# test_df = test_df.merge(overview_df, on="title", how="left")

# Fill missing values
for col in ["user_review", "overview", "tags", "developer", "publisher"]:
    if col in train_df.columns:
        train_df[col] = train_df[col].fillna("")
    if col in test_df.columns:
        test_df[col] = test_df[col].fillna("")


# 3. Build text field
#    You can use only user_review, or combine fields

train_df["full_text"] = (
    train_df["user_review"]
)

test_df["full_text"] = (
    test_df["user_review"]
)


# 4. Define features and labels

X = train_df["full_text"]
y = train_df["user_suggestion"]

X_unlabeled_test = test_df["full_text"]   # no y_test exists in this file


# 5. Create validation split from training data
#    Use this for accuracy / F1

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# 6. TF-IDF vectorization

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_unlabeled_test)


# 7. Logistic Regression with tuning

param_grid = {
    "C": [0.01, 0.1, 1, 10]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=2000, class_weight="balanced"),
    param_grid,
    cv=3,
    scoring="f1_weighted",
    n_jobs=-1
)

grid.fit(X_train_tfidf, y_train)

best_model = grid.best_estimator_

print("Best parameters:", grid.best_params_)


# 8. Validation predictions and metrics

y_val_pred = best_model.predict(X_val_tfidf)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Validation F1 Score (weighted):", f1_score(y_val, y_val_pred, average="weighted"))
print("\nClassification Report:\n")
print(classification_report(y_val, y_val_pred))


# 9. Top positive / negative features

feature_names = vectorizer.get_feature_names_out()
coefficients = best_model.coef_[0]

top_positive_idx = np.argsort(coefficients)[-10:]
top_negative_idx = np.argsort(coefficients)[:10]

print("\nTop Positive Features:")
for i in reversed(top_positive_idx):
    print(f"{feature_names[i]}: {coefficients[i]:.4f}")

print("\nTop Negative Features:")
for i in top_negative_idx:
    print(f"{feature_names[i]}: {coefficients[i]:.4f}")


# 10. show top positive/negative bigrams only

positive_bigrams = [(feature_names[i], coefficients[i]) for i in top_positive_idx if " " in feature_names[i]]
negative_bigrams = [(feature_names[i], coefficients[i]) for i in top_negative_idx if " " in feature_names[i]]

print("\nTop Positive Bigrams:")
for phrase, weight in sorted(positive_bigrams, key=lambda x: x[1], reverse=True):
    print(f"{phrase}: {weight:.4f}")

print("\nTop Negative Bigrams:")
for phrase, weight in sorted(negative_bigrams, key=lambda x: x[1]):
    print(f"{phrase}: {weight:.4f}")


# 11. inspect misclassified validation examples

val_results = pd.DataFrame({
    "text": X_val.values,
    "true_label": y_val.values,
    "pred_label": y_val_pred
})

misclassified = val_results[val_results["true_label"] != val_results["pred_label"]]

print("\nSample Misclassified Reviews:")
print(misclassified.head(5))

# 12. Train final model on ALL labeled training data
#    and generate predictions for unlabeled test set

X_all_tfidf = vectorizer.fit_transform(X)
X_test_final_tfidf = vectorizer.transform(X_unlabeled_test)

final_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    C=grid.best_params_["C"]
)

final_model.fit(X_all_tfidf, y)

test_predictions = final_model.predict(X_test_final_tfidf)
test_probabilities = final_model.predict_proba(X_test_final_tfidf)[:, 1]


# 13. Save predictions for the unlabeled test set

submission = pd.DataFrame({
    "review_id": test_df["review_id"],
    "predicted_user_suggestion": test_predictions,
    "probability_recommend": test_probabilities
})

submission.to_csv("logistic_regression_predictions.csv", index=False)

print("\nSaved predictions to logistic_regression_predictions.csv")

Best parameters: {'C': 10}
Validation Accuracy: 0.8562446413260931
Validation F1 Score (weighted): 0.8562387727343461

Classification Report:

              precision    recall  f1-score   support

           0       0.83      0.83      0.83      1505
           1       0.87      0.87      0.87      1994

    accuracy                           0.86      3499
   macro avg       0.85      0.85      0.85      3499
weighted avg       0.86      0.86      0.86      3499


Top Positive Features:
best: 7.7575
addictive: 7.5925
amazing: 6.9816
great: 6.7577
alpha: 6.5534
definitely: 5.7018
awesome: 5.6561
fantastic: 5.6513
love: 5.6226
highly: 5.5553

Top Negative Features:
worst: -9.4973
ruined: -8.1503
terrible: -7.6408
poor: -7.6142
worse: -7.4991
unplayable: -6.8231
awful: -6.7574
boring: -6.5541
money: -6.5389
garbage: -6.5097

Top Positive Bigrams:

Top Negative Bigrams:

Sample Misclassified Reviews:
                                                 text  true_label  pred_label
1   Early 